# bnr_lab-3

In [163]:
from pyspark.sql import functions as F
from pyspark.sql import Row
from pyspark.sql.types import *
from datetime import date

In [2]:
spark

In [156]:
schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("date", DateType(), True),
    StructField("dates_active", ArrayType(DateType()), True)
])

df_users_cumulated = spark.createDataFrame(
    [
        Row(user_id = 1,
            date = date(2023, 1, 30),
            dates_active = [date(2023, 1, 30), date(2023, 1, 28), date(2023, 1, 1)]),
    ],
    schema=schema
)

In [157]:
df_users_cumulated.createOrReplaceTempView("users_cumulated")

In [158]:
spark.sql("select * from users_cumulated").show()

+-------+----------+--------------------+
|user_id|      date|        dates_active|
+-------+----------+--------------------+
|      1|2023-01-30|[2023-01-30, 2023...|
+-------+----------+--------------------+



In [195]:
query = """
    with a as (
        select * 
        from users_cumulated
        where date = '2023-01-30'
    ),
    b as (
        select 
        explode(sequence(to_date('2023-01-01'), to_date('2023-01-30'), interval '1 day')) as generate_series
    ),
    c as (
        select
            *,
            case
                when array_contains(dates_active, generate_series)
                    then pow(2, 32 - cast((date - generate_series) as int))
                else 0
            end as datelist_pow2
        from a 
        cross join b
    ),
    d as (
        select 
            user_id,
            min(date) as date,
            min(dates_active) as dates_active, -- just for referece, we would not actually keep this one
            lpad(bin(sum(datelist_pow2)), 33, 0) as datelist
        from c
        group by user_id
    )
    select * from d
"""

In [185]:
spark.sql("select bin(pow(2,8))").show()

+--------------+
|bin(pow(2, 8))|
+--------------+
|     100000000|
+--------------+



In [194]:
spark.sql(query).show(truncate=False)

+-------+----------+------------------------------------+---------------------------------+
|user_id|date      |dates_active                        |datelist                         |
+-------+----------+------------------------------------+---------------------------------+
|1      |2023-01-30|[2023-01-30, 2023-01-28, 2023-01-01]|010100000000000000000000000000100|
+-------+----------+------------------------------------+---------------------------------+



In [196]:
spark.sql(query).show(truncate=False)

+-------+----------+------------------------------------+---------------------------------+
|user_id|date      |dates_active                        |datelist                         |
+-------+----------+------------------------------------+---------------------------------+
|1      |2023-01-30|[2023-01-30, 2023-01-28, 2023-01-01]|101000000000000000000000000001000|
+-------+----------+------------------------------------+---------------------------------+



In [160]:
spark.sql(query).show(truncate=False)

+-------+----------+------------------------------------+--------------------------------+
|user_id|date      |dates_active                        |datelist                        |
+-------+----------+------------------------------------+--------------------------------+
|1      |2023-01-30|[2023-01-30, 2023-01-28, 2023-01-01]|10100000000000000000000000000100|
+-------+----------+------------------------------------+--------------------------------+



In [161]:
spark.sql(query).printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- dates_active: array (nullable = true)
 |    |-- element: date (containsNull = true)
 |-- datelist: string (nullable = true)



In [166]:
schema_expected = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("date", DateType(), True),
    StructField("dates_active", ArrayType(DateType()), True),
    StructField("datelist", StringType(), True)
])

df_expected = spark.createDataFrame(
    [
        Row(user_id = 1,
            date = date(2023, 1, 30),
            dates_active = [date(2023, 1, 30), date(2023, 1, 28), date(2023, 1, 1)],
            datelist = '10100000000000000000000000000100',
           )
    ],
    schema=schema_expected
)

In [ ]:
d as (
        select 
            user_id,
            min(date) as date,
            min(dates_active) as dates_active, -- just for referece, we would not actually keep this one
            lpad(bin(sum(datelist_pow2)), 32, 0) as datelist
        from c
        group by user_id
    )
    select * from d

In [ ]:
query = """
    with a as (
        select * 
        from users_cumulated
        where date = '2023-01-30'
    ),
    b as (
        select generate_series('2023-01-01', '2023-01-30', interval '1 day')
    ),
    c as (
        select
        *,
        case
            when dates_active @> array[generate_series::date]
                then pow(2, 32 - (date-generate_series::date) -1)
                -- note that the code from zach was missing this -1
                -- we nee thids.
                -- otherwise we will not be tracking wether the current_date was active or not
            else 0
        end as datelist_pow2
        from a 
        cross join b
    ),
    d as (
        select 
        user_id,
        min(date) as date,
        min(dates_active) as dates_active, -- just for referece, we would not actually keep this one
        sum(datelist_pow2)::bigint::bit(32) as datelist
        from c
        group by user_id
    )
    select
    *,
    bit_count(datelist)>0 as dim_monthly_active,
    bit_count('11111110000000000000000000000000'::bit(32) & datelist)>0 as dim_weekly_active,
    bit_count('10000000000000000000000000000000'::bit(32) & datelist)>0 as dim_daily_active
    from d
    ;
"""

+---+
| id|
+---+
|  1|
+---+

